## Environment setup

In [1]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

### Install dependencies

Installs all required Python libraries, including [`google-generativeai`](https://pypi.org/project/google-generativeai/) for Gemini and [`supervision`](https://github.com/roboflow/supervision) for output parsing and visualization.


In [2]:
!pip install google-genai supervision

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.2/88.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.0/385.0 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.3/144.3 kB 6.5 MB/s eta 0:00:00


In [3]:
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

safety_settings = [
    types.SafetySetting(
        category="HARM_CATEGORY_DANGEROUS_CONTENT",
        threshold="BLOCK_ONLY_HIGH",
    ),
]

## Detect dents across multiple images

Runs one Gemini API call per image (`dent`, `dent1`, `dent2`, `dent3`), parses the returned bounding box, and renders it directly on that image.

In [4]:
from PIL import Image
import supervision as sv

MODEL_NAME = "gemini-3.7-flash"
TEMPERATURE = 0.7

PROMPT = "Detect any dent on this car. Return only the 2D bounding box coordinates of the dent as [x1, y1, x2, y2, ..., xn, yn]."

IMAGE_NAMES = ["dent", "dent1", "dent2", "dent3"]
IMAGE_PATHS = [f"/content/{name}.jpeg" for name in IMAGE_NAMES]

In [5]:
annotated_images = []

for name, image_path in zip(IMAGE_NAMES, IMAGE_PATHS):
    image = Image.open(image_path)
    width, height = image.size
    target_height = int(1024 * height / width)
    resized_image = image.resize((1024, target_height), Image.Resampling.LANCZOS)

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=[resized_image, PROMPT],
        config=types.GenerateContentConfig(
            temperature=TEMPERATURE,
            safety_settings=safety_settings,
            thinking_config=types.ThinkingConfig(
                thinking_budget=0
            )
        )
    )

    resolution_wh = image.size
    detections = sv.Detections.from_vlm(
        vlm=sv.VLM.GOOGLE_GEMINI_2_5,
        result=response.text,
        resolution_wh=resolution_wh
    )

    thickness = sv.calculate_optimal_line_thickness(resolution_wh=resolution_wh)
    text_scale = sv.calculate_optimal_text_scale(resolution_wh=resolution_wh)

    box_annotator = sv.BoxAnnotator(thickness=thickness)
    label_annotator = sv.LabelAnnotator(
        smart_position=True,
        text_color=sv.Color.BLACK,
        text_scale=text_scale,
        text_position=sv.Position.CENTER
    )

    annotated = image
    for annotator in (box_annotator, label_annotator):
        annotated = annotator.annotate(scene=annotated, detections=detections)

    annotated_images.append((name, annotated))
    print(f"{name}: {response.text}")

FileNotFoundError: [Errno 2] No such file or directory: '/content/dent.jpeg'

## Render results

In [ ]:
for name, annotated in annotated_images:
    print(name)
    sv.plot_image(annotated)